# 02c. Treinar Splink — mãe + endereço (sem UF)

Modelo paralelo ao [`02_treinar_splink.ipynb`](02_treinar_splink.ipynb).

Comparisons: `nome_completo_phon` e `data_nascimento` iguais ao 02;
`nome_mae_phon` (com nível Contida); `endereco` composto
(logradouro / CEP / município). **Sem UF** no score.

Grava `splink_model_mae_endereco.json`. Predict:
[`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb).
Atribuição simples:
[`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb).


In [1]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

SPLINK_MODEL_JSON = SPLINK_MODEL_MAE_ENDERECO

print_paths()
con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
U_MAX_PAIRS = 500_000_000
EM_MAX_PAIRS = 10_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(
        f'Amostra só para profile (não entra no Linker): '
        f'{SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}'
    )


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

splink_input → censo_limpo ∪ cpf_limpo


Registros: 15,036,974
DuckDB: threads=20, memory_limit=344.5 GiB (defaults: 20, 370GB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Amostra só para profile (não entra no Linker): 1,000,000 de 15,036,974


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.


In [2]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'primeiro_ultimo',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
    'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'primeiro_ultimo_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


,origem,n
0,cpf,8447170
1,censo,6589804


,0,1
origem,censo,cpf
pct_primeiro_nome,100.0,100.0
pct_ultimo_nome,99.6,100.0
pct_nome_completo,100.0,100.0
pct_nome_meio,88.4,96.0
pct_primeiro_ultimo,100.0,100.0
pct_nome_mae,33.8,96.9
pct_primeiro_nome_mae,33.8,96.9
pct_nome_meio_mae,30.0,90.6
pct_ultimo_nome_mae,33.8,96.9


## Exploração pré-modelo

Profile Splink das colunas de linkage (amostra se a base passar de
`SPLINK_ANALYSIS_SAMPLE_N`). O treino usa o conjunto inteiro. Volume de
candidatos do `predict` é o gráfico de blocking no
[`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb).


In [3]:
from splink.exploratory import profile_columns

profile_cols = [
    'nome_mae_phon', 'logradouro_norm', 'cep', 'cod_municipio',
    'primeiro_nome_phon', 'nome_meio_phon', 'ultimo_nome_phon', 'nome_completo_phon',
    'primeiro_ultimo_phon',
    'uf', 'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
    'idade', 'cpf_norm',
]
available = set(con.execute(f'SELECT * FROM {analysis_table} LIMIT 0').df().columns)
ausentes = [c for c in profile_cols if c not in available]
profile_cols = [c for c in profile_cols if c in available]
if ausentes:
    print('Profile: colunas ausentes (rode o 00 se quiser o composto):', ausentes)

profile_columns(
    db_api.register(analysis_table),
    column_expressions=profile_cols,
)


SplinkChart (use .to_html() or .save() to export HTML)

## Modelo Splink (mãe + endereço)

`Linker` nas views `splink_censo` / `splink_cpf` = limpo cheio (treino).
Quatro comparisons; sem UF.


In [4]:
import splink.comparison_level_library as cll
import splink.comparison_library as cl

JW_MAE_ALTO = 0.92
JW_MAE_MEDIO = 0.85
JW_LOGR_SIMILAR = 0.90

null_sql = (
    '(data_nascimento_l IS NULL OR data_nascimento_r IS NULL) AND '
    '(idade_l IS NULL OR idade_r IS NULL) AND '
    '(mes_nascimento_l IS NULL OR mes_nascimento_r IS NULL OR '
    'dia_nascimento_l IS NULL OR dia_nascimento_r IS NULL)'
)
dob_presente_sql = (
    'data_nascimento_l IS NOT NULL AND data_nascimento_r IS NOT NULL'
)
troca_mes_dia_sql = (
    'ano_nascimento_l = ano_nascimento_r AND '
    'mes_nascimento_l = dia_nascimento_r AND '
    'dia_nascimento_l = mes_nascimento_r AND '
    'mes_nascimento_l <> dia_nascimento_l'
)
dia_diferente_sql = (
    'ano_nascimento_l = ano_nascimento_r AND '
    'mes_nascimento_l = mes_nascimento_r AND '
    'dia_nascimento_l <> dia_nascimento_r'
)
mes_diferente_sql = (
    'ano_nascimento_l = ano_nascimento_r AND '
    'dia_nascimento_l = dia_nascimento_r AND '
    'mes_nascimento_l <> mes_nascimento_r'
)
ano_pm1_sql = (
    'mes_nascimento_l = mes_nascimento_r AND '
    'dia_nascimento_l = dia_nascimento_r AND '
    'abs(try_cast(ano_nascimento_l AS INTEGER) - '
    'try_cast(ano_nascimento_r AS INTEGER)) = 1'
)

split_l = "string_split(nome_completo_phon_l, ' ')"
split_r = "string_split(nome_completo_phon_r, ' ')"
n_min = f'least(len({split_l}), len({split_r}))'
prefixo_sql = (
    f'{n_min} >= 2 AND len({split_l}) <> len({split_r}) AND '
    f'list_slice({split_l}, 1, {n_min}) = list_slice({split_r}, 1, {n_min})'
)
jw_ultimo_095_sql = (
    f"{split_l}[-1] = {split_r}[-1] AND "
    'jaro_winkler_similarity(nome_completo_phon_l, nome_completo_phon_r) >= 0.95'
)
jw_ultimo_092_sql = (
    f"{split_l}[-1] = {split_r}[-1] AND "
    'jaro_winkler_similarity(nome_completo_phon_l, nome_completo_phon_r) >= 0.92'
)
dl_completo_1_sql = (
    'damerau_levenshtein(nome_completo_phon_l, nome_completo_phon_r) <= 1'
)
n_tok = f'len({split_l})'
mesmo_n = f'{n_tok} = len({split_r}) AND {n_tok} >= 3'
zip_tok = f'list_zip({split_l}, {split_r})'
dif_count = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    'x -> CASE WHEN x[1] <> x[2] THEN 1 ELSE 0 END'
    "), 'sum')"
)
dif_max_dl = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    'x -> CASE WHEN x[1] <> x[2] '
    'THEN damerau_levenshtein(x[1], x[2]) ELSE 0 END'
    "), 'max')"
)
um_token_sql = (
    f'{mesmo_n} AND {split_l}[-1] = {split_r}[-1] '
    f'AND {dif_count} = 1 AND {dif_max_dl} <= 2'
)
dois_tokens_sql = (
    f'{mesmo_n} AND {dif_count} BETWEEN 1 AND 2 AND {dif_max_dl} <= 2'
)
n_l = f'len({split_l})'
n_r = f'len({split_r})'
ordem_sql = (
    f'{n_l} = {n_r} AND {n_l} >= 3 '
    f'AND list_sort({split_l}) = list_sort({split_r}) '
    f'AND {split_l} <> {split_r}'
)

def _sem_token(split, k):
    n = f'len({split})'
    if k == 1:
        return f'list_slice({split}, 2, {n})'
    return (
        f'list_concat(list_slice({split}, 1, {k - 1}), '
        f'list_slice({split}, {k + 1}, {n}))'
    )

_ors_token = []
for _k in range(1, 11):
    _ors_token.append(f'({n_l} >= {_k} AND {_sem_token(split_l, _k)} = {split_r})')
    _ors_token.append(f'({n_r} >= {_k} AND {_sem_token(split_r, _k)} = {split_l})')
token_a_mais_sql = (
    f'abs({n_l} - {n_r}) = 1 AND least({n_l}, {n_r}) >= 3 AND ('
    + ' OR '.join(_ors_token)
    + ')'
)

# --- mãe ---
split_mae_l = "string_split(nome_mae_phon_l, ' ')"
split_mae_r = "string_split(nome_mae_phon_r, ' ')"
n_mae = f'least(len({split_mae_l}), len({split_mae_r}))'
mae_contem_sql = (
    f'{n_mae} >= 2 AND '
    f'len({split_mae_l}) <> len({split_mae_r}) AND '
    f'len(list_intersect({split_mae_l}, {split_mae_r})) = {n_mae}'
)
mae_ordem_sql = (
    f'len({split_mae_l}) = len({split_mae_r}) AND len({split_mae_l}) >= 3 '
    f'AND list_sort({split_mae_l}) = list_sort({split_mae_r}) '
    f'AND nome_mae_phon_l <> nome_mae_phon_r'
)
mae_jw_alto_sql = (
    f'jaro_winkler_similarity(nome_mae_phon_l, nome_mae_phon_r) >= {JW_MAE_ALTO}'
)
mae_jw_medio_sql = (
    f'jaro_winkler_similarity(nome_mae_phon_l, nome_mae_phon_r) >= {JW_MAE_MEDIO}'
)

# --- endereço composto ---
endereco_null_sql = (
    '(logradouro_norm_l IS NULL OR logradouro_norm_r IS NULL) AND '
    '(cep_l IS NULL OR cep_r IS NULL) AND '
    '(cod_municipio_l IS NULL OR cod_municipio_r IS NULL)'
)
logr_cep_exact_sql = (
    'logradouro_norm_l = logradouro_norm_r AND cep_l = cep_r'
)
logr_mun_exact_sql = (
    'logradouro_norm_l = logradouro_norm_r AND '
    'cod_municipio_l = cod_municipio_r'
)
logr_sim_cep_sql = (
    f'jaro_winkler_similarity(logradouro_norm_l, logradouro_norm_r) '
    f'>= {JW_LOGR_SIMILAR} AND cep_l = cep_r'
)
cep_exact_sql = 'cep_l = cep_r'

comparisons = [
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('nome_completo_phon'),
            cll.ExactMatchLevel(
                'nome_completo_phon', term_frequency_adjustments=True
            ),
            cll.CustomLevel(prefixo_sql, label_for_charts='Prefixo de tokens'),
            cll.CustomLevel(dl_completo_1_sql, label_for_charts='DL <= 1'),
            cll.CustomLevel(um_token_sql, label_for_charts='Um token DL <= 2'),
            cll.CustomLevel(
                dois_tokens_sql, label_for_charts='Ate dois tokens DL <= 2'
            ),
            cll.CustomLevel(
                token_a_mais_sql, label_for_charts='Um token a mais'
            ),
            cll.CustomLevel(
                ordem_sql, label_for_charts='Mesmos tokens, ordem diferente'
            ),
            cll.CustomLevel(
                jw_ultimo_095_sql, label_for_charts='Ultimo token e JW >= 0.95'
            ),
            cll.CustomLevel(
                jw_ultimo_092_sql, label_for_charts='Ultimo token e JW >= 0.92'
            ),
            cll.ElseLevel(),
        ],
        output_column_name='nome_completo_phon',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.CustomLevel(null_sql, label_for_charts='Null data e mes/dia').configure(
                is_null_level=True
            ),
            cll.ExactMatchLevel('data_nascimento', term_frequency_adjustments=True),
            cll.CustomLevel(
                troca_mes_dia_sql, label_for_charts='Mes e dia trocados'
            ),
            cll.CustomLevel(
                dia_diferente_sql, label_for_charts='Dia diferente'
            ),
            cll.CustomLevel(
                mes_diferente_sql, label_for_charts='Mes diferente'
            ),
            cll.CustomLevel(ano_pm1_sql, label_for_charts='Ano +/- 1'),
            cll.DamerauLevenshteinLevel('data_nascimento', 1),
            cll.DamerauLevenshteinLevel('data_nascimento', 2),
            cll.CustomLevel(
                dob_presente_sql,
                label_for_charts='Datas presentes, demais diferenças',
            ),
            cll.ExactMatchLevel('idade'),
            cll.AbsoluteDifferenceLevel('idade', 1),
            cll.ElseLevel(),
        ],
        output_column_name='data_nascimento',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('nome_mae_phon'),
            cll.ExactMatchLevel(
                'nome_mae_phon', term_frequency_adjustments=True
            ),
            cll.CustomLevel(mae_contem_sql, label_for_charts='Contida'),
            cll.CustomLevel(
                mae_ordem_sql, label_for_charts='Mesmos tokens, ordem diferente'
            ),
            cll.CustomLevel(
                mae_jw_alto_sql, label_for_charts=f'JW >= {JW_MAE_ALTO}'
            ),
            cll.CustomLevel(
                mae_jw_medio_sql, label_for_charts=f'JW >= {JW_MAE_MEDIO}'
            ),
            cll.ElseLevel(),
        ],
        output_column_name='nome_mae_phon',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.CustomLevel(
                endereco_null_sql, label_for_charts='Null / nao comparavel'
            ).configure(is_null_level=True),
            cll.CustomLevel(
                logr_cep_exact_sql, label_for_charts='Logradouro e CEP exact'
            ),
            cll.CustomLevel(
                logr_mun_exact_sql,
                label_for_charts='Logradouro e municipio exact',
            ),
            cll.CustomLevel(
                logr_sim_cep_sql,
                label_for_charts='Logradouro similar e CEP exact',
            ),
            cll.CustomLevel(cep_exact_sql, label_for_charts='CEP exact'),
            cll.ElseLevel(),
        ],
        output_column_name='endereco',
    ),
]


In [5]:
from splink import Linker, SettingsCreator

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}').fetchone()[0]
n_cpf_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}').fetchone()[0]
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
assert n_censo_view == n_censo_limpo and n_cpf_view == n_cpf_limpo, (
    f'Linker não está no conjunto inteiro: '
    f'censo {n_censo_view} vs limpo {n_censo_limpo}; '
    f'cpf {n_cpf_view} vs limpo {n_cpf_limpo}'
)
print('Linker: conjunto inteiro', f'{n_censo_view:,}', f'{n_cpf_view:,}')

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    retain_intermediate_calculation_columns=False,
)
censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'
linker = Linker([censo_in, cpf_in], settings)


Linker: conjunto inteiro 6,589,804 8,447,170


## Prior

Outra lista, distinta da predição (02b): `nome_completo+DOB` e `cpf_norm`. Estima a
probabilidade a priori de dois registros aleatórios casarem. O `u` vem de
pares **aleatórios** (`U_MAX_PAIRS`).


In [6]:
from splink import block_on

deterministic_rules = [
    block_on('cpf_norm'),
    block_on('nome_completo_phon', 'data_nascimento'),    
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
# Amostra de pares aleatórios para o parâmetro u (não são os pares do blocking).
print('max_pairs u:', f'{U_MAX_PAIRS:,}')
linker.training.estimate_u_using_random_sampling(max_pairs=U_MAX_PAIRS)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Probability two random records match is estimated to be  8.05e-08.
This means that amongst all possible pairwise record comparisons, one in 12,429,820.83 are expected to match.  With 55,665,194,654,680 total possible comparisons, we expect a total of around 4,478,358.57 matching pairs


----- Estimating u probabilities using random sampling -----


Estimating u with: max_pairs = 500,000,000, min_count_per_level = 100, num_chunks = 10


max_pairs u: 500,000,000


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Count of 387 for level Logradouro similar e CEP exact (cvv=2). Chunk took 7.0 seconds.


  Exiting early since min count of 387 exceeds min_count_per_level = 100



Estimated u probabilities using random sampling



Your model is not yet fully trained. Missing estimates for:
    - nome_completo_phon (no m values are trained).
    - data_nascimento (no m values are trained).
    - nome_mae_phon (no m values are trained).
    - endereco (no m values are trained).


## EM

Mesma arquitetura do 02: `u` fixo da amostra aleatória; `m` via `cpf_norm`;
EM nome (data+sexo); EM data (primeiro+último); **mais** EM mãe
(nome completo+data) e EM endereço (primeiro+último+data).


In [7]:
# m direto: mesmo cpf_norm é match. NULL não casa com NULL.
linker.training.estimate_m_from_label_column('cpf_norm')

# Nome. A data está na chave, então a comparison temporal sai desta sessão.
em_nome = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('data_nascimento', 'sexo'),
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)

# Data. Primeiro+último não desliga nome_completo; a condição do nome
# é sempre verdadeira e só tira essa comparison da sessão.
bloqueio_data = (
    'l.primeiro_nome_phon = r.primeiro_nome_phon AND '
    'l.ultimo_nome_phon = r.ultimo_nome_phon AND '
    '(l.nome_completo_phon IS NULL OR l.nome_completo_phon IS NOT NULL)'
)
em_data = linker.training.estimate_parameters_using_expectation_maximisation(
    bloqueio_data,
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)
m_data = linker._settings_obj.core_model_settings.copy()

# Diagnóstico: o mesmo bloco com sexo. Não fica no modelo salvo.
bloqueio_data_sexo = bloqueio_data + ' AND l.sexo = r.sexo'
em_data_sexo = linker.training.estimate_parameters_using_expectation_maximisation(
    bloqueio_data_sexo,
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)
linker._settings_obj.core_model_settings = m_data
linker._populate_m_u_from_trained_values()

# Mãe: nome+data na chave → comparison de mãe treina.
em_mae = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('nome_completo_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)

# Endereço: primeiro+último+data (endereço fora da chave).
em_endereco = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

------- Estimating m probabilities using from column cpf_norm --------


m probability not trained for nome_completo_phon - Prefixo de tokens (comparison vector value: 8). This usually means the comparison level was never observed in the training data.


m probability not trained for nome_completo_phon - DL <= 1 (comparison vector value: 7). This usually means the comparison level was never observed in the training data.


...

Iteration 5: Largest change in params was 0.000651 in the m_probability of nome_completo_phon, level `All other comparisons`


Iteration 6: Largest change in params was 0.000233 in the m_probability of nome_completo_phon, level `All other comparisons`


Iteration 7: Largest change in params was 8.34e-05 in the m_probability of nome_completo_phon, level `All other comparisons`



EM converged after 7 iterations



Your model is fully trained. All comparisons have at least one estimate for their m and u values


In [8]:
# O treino de m está na célula anterior.

In [9]:
# O treino de m está na célula do cpf_norm.

In [10]:
# O treino de m está na célula do cpf_norm.

In [11]:
# O treino de m está na célula do cpf_norm.

In [12]:
from config import MODELS_DIR

SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
linker.misc.save_model_to_json(str(MODELS_DIR / 'splink_model_mae_endereco.json'), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)
print('Cópia em models/:', MODELS_DIR / 'splink_model_mae_endereco.json')


Modelo salvo: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_model_mae_endereco.json
Cópia em models/: /home/ibge.gov.br/ramon.goncalves/ibge-probabilistico/models/splink_model_mae_endereco.json


## Visualização pós-treino

Match weights do modelo salvo e o `m` das sessões de EM lado a lado
(nome, data, data+sexo). O bloco com sexo não entra no JSON.


In [13]:
linker.visualisations.match_weights_chart()


SplinkChart (use .to_html() or .save() to export HTML)

In [14]:
linker.visualisations.parameter_estimate_comparisons_chart()


SplinkChart (use .to_html() or .save() to export HTML)

In [15]:
linker.evaluation.unlinkables_chart()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SplinkChart (use .to_html() or .save() to export HTML)

## Encerrar

JSON pronto para o
[`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb).


In [16]:
print(f"{'modelo':12s} {'ok ' if SPLINK_MODEL_JSON.exists() else 'FALTA'} {SPLINK_MODEL_JSON}")


modelo       ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_model_mae_endereco.json


In [17]:
con.close()
